# One Health Disease Intelligence: Spatiotemporal Risk Analysis & Epidemic Modeling

This notebook implements the second phase of our early warning disease intelligence workflow:
1. **Evaluate spatiotemporal outbreak probabilities** using our `RiskAnalysisAgent`'s logistic-sigmoid scoring engine.
2. **Simulate disease transmission trajectories ($R_0$)** and **Herd Immunity Thresholds ($HIT$)** using the compartmental `EpidemiologicalModelingEngine` to evaluate the impact of targeted control interventions.
3. **Export the processed outcome database** to feed our interactive R Shiny web dashboard.

In [ ]:
# 1. Initialize Python Path for local module imports
import os
import sys
import pandas as pd
import numpy as np

# Ensure the 'src/' directory is in our path so we can import local modules
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../src")))
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

print("System path initialized successfully.")

## Step 2: Load the Cleaned Master Dataset
We load the merged and validated database compiled during our first ingestion step.

In [ ]:
cleaned_data_path = "../data/processed/cleaned_master_dataset.csv"

if os.path.exists(cleaned_data_path):
    df_cleaned = pd.read_csv(cleaned_data_path)
    print(f"[SUCCESS] Loaded cleaned master dataset containing {len(df_cleaned)} records.")
else:
    print("[ERROR] Missing cleaned dataset. Run the 01_data_ingestion.ipynb notebook first.")

## Step 3: Run the Spatiotemporal Risk Assessment Engine
We use the `RiskAnalysisAgent` to compute disease outbreak probabilities on a district/woreda level, assigning a risk classification of *Low*, *Medium*, *High*, or *Very High Risk*.

In [ ]:
from agents.risk_analysis_agent import RiskAnalysisAgent

# Initialize Risk Analysis Agent
analyzer = RiskAnalysisAgent()

# Run evaluation
df_scored = analyzer.evaluate_risk_matrix(df_cleaned, df_cleaned)

# Save output for our web dashboard
output_path = "../data/processed/risk_assessment_output.csv"
df_scored.to_csv(output_path, index=False)
print(f"Calculated disease probabilities and saved outcomes to: {output_path}")

## Step 4: Run the Epidemiological Transmission Engine (R0 Calculations)
Using the `EpidemiologicalModelingEngine`, we calculate the Basic Reproduction Number ($R_0$) and Herd Immunity Thresholds ($HIT$) across our active districts. This allows us to compare low-intervention scenarios with optimal targeted control campaigns.

In [ ]:
from agents.epidemiological_modeling_engine import EpidemiologicalModelingEngine

model_engine = EpidemiologicalModelingEngine()

# Isolate a specific district profile (e.g. Jinka town)
jinka_data = df_scored[df_scored["district"] == "Jinka"].iloc[0]

# Case A: Low control measures (45% vaccination, no active isolation)
low_intervention = model_engine.calculate_r0(
    transmission_rate=2.95,
    susceptible_population=int(jinka_data["population"]),
    total_population=287502,
    vaccination_coverage=0.45,
    isolation_rate=0.0
)

# Case B: Optimal control measures (94% vaccination, active isolation)
optimal_intervention = model_engine.calculate_r0(
    transmission_rate=2.95,
    susceptible_population=int(jinka_data["population"]),
    total_population=287502,
    vaccination_coverage=0.94,
    isolation_rate=0.259
)

print("=== SURVEILLANCE & THRESHOLD MODELLING RESULTS ===\n")
print(f"Jinka Baseline R0 (45% Vax): {low_intervention['r0']} -> {low_intervention['epidemiological_status']}")
print(f"  Required Herd Immunity: {low_intervention['herd_immunity_threshold_pct']}%\n")
print(f"Jinka Targeted R0 (94% Vax + Isolation): {optimal_intervention['r0']} -> {optimal_intervention['epidemiological_status']}")
print(f"  Required Herd Immunity: {optimal_intervention['herd_immunity_threshold_pct']}%\n")